# 01 — Data collection & cleaning (Alzheimer's disease target, ChEMBL)

Run every cell top to bottom in Google Colab. No local install needed — this runs entirely on Google's servers, so it works from any browser, including older macOS versions.

**Default target: acetylcholinesterase (AChE)** — `CHEMBL220`, the confirmed ChEMBL ID for human AChE, the enzyme targeted by every currently approved Alzheimer's drug (donepezil, rivastigmine, galantamine). Cholinesterase inhibition remains one of the two most tractable, well-validated AD drug targets for a docking/QSAR-style workflow (the other being BACE1, which the lookup cell below can find for you).

**What this notebook does:**
1. (Optional) Looks up the correct ChEMBL target ID for any Alzheimer's target by name, so you never have to guess or hardcode a wrong ID.
2. Downloads real bioactivity data (IC50 values) for the chosen target directly from ChEMBL.
3. Cleans and deduplicates it.
4. Computes RDKit molecular descriptors for every compound.
5. Saves a single clean CSV you'll use in notebook 2.


In [ ]:
# Install dependencies (Colab-safe versions)
!pip install -q chembl_webresource_client rdkit pandas numpy


In [ ]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen


### Optional: look up a target's ChEMBL ID by name

Alzheimer's disease has several well-studied targets. Run this cell to search
for any of them by name and confirm the exact `target_chembl_id` before using
it below — this avoids ever hardcoding a wrong ID. A few starting points:
- `Acetylcholinesterase` — cholinesterase inhibition (default here, `CHEMBL220`)
- `Butyrylcholinesterase` — secondary cholinesterase target
- `Beta-secretase 1` (BACE1) — amyloid-beta production
- `Glycogen synthase kinase-3 beta` (GSK-3β) — tau hyperphosphorylation


In [ ]:
def find_target(name_query, organism="Homo sapiens"):
    hits = new_client.target.filter(
        pref_name__icontains=name_query, organism=organism
    ).only("target_chembl_id", "pref_name", "organism", "target_type")
    return pd.DataFrame(list(hits))

# Example: search for BACE1 candidates -- inspect the result, then copy the
# target_chembl_id you want into TARGET_CHEMBL_ID in the next cell.
find_target("secretase 1")


In [ ]:
# ---- CONFIG ----
TARGET_CHEMBL_ID = "CHEMBL220"   # human acetylcholinesterase (AChE) -- confirmed ID
MAX_RECORDS = 3000                # cap for a manageable demo dataset


In [ ]:
# 1. Pull bioactivity records (IC50, nM, human assays) for the target
activity = new_client.activity
records = activity.filter(
    target_chembl_id=TARGET_CHEMBL_ID,
    standard_type="IC50",
    standard_units="nM",
    assay_type="B",
).only(
    "canonical_smiles", "standard_value", "standard_units",
    "standard_type", "molecule_chembl_id", "assay_description"
)

rows = []
for i, r in enumerate(records):
    if i >= MAX_RECORDS:
        break
    rows.append(r)

df_raw = pd.DataFrame(rows)
print(f"Fetched {len(df_raw)} raw records")
df_raw.head()


In [ ]:
# 2. Clean: drop missing values, invalid SMILES, and non-numeric potency values
df = df_raw.dropna(subset=["canonical_smiles", "standard_value"]).copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"])
df = df[df["standard_value"] > 0]

def valid_smiles(smi):
    return Chem.MolFromSmiles(smi) is not None

df = df[df["canonical_smiles"].apply(valid_smiles)]

# Deduplicate: same compound tested multiple times -> take the median IC50
df = (
    df.groupby(["molecule_chembl_id", "canonical_smiles"], as_index=False)["standard_value"]
    .median()
)

print(f"After cleaning and dedup: {len(df)} unique compounds")


In [ ]:
# 3. Convert IC50 (nM) to pIC50 (standard QSAR target variable)
# pIC50 = -log10(IC50 in molar) = 9 - log10(IC50 in nM)
df["pIC50"] = 9 - np.log10(df["standard_value"])
df = df[np.isfinite(df["pIC50"])]


In [ ]:
# 4. Compute RDKit descriptors for every compound
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.Series([np.nan]*8)
    return pd.Series([
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Descriptors.TPSA(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
        Lipinski.NumAromaticRings(mol),
    ])

desc_cols = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds", "RingCount", "AromaticRings"]
df[desc_cols] = df["canonical_smiles"].apply(compute_descriptors)
df = df.dropna(subset=desc_cols)
print(f"Final dataset: {len(df)} compounds x {len(desc_cols)} descriptors")
df.head()


In [ ]:
# 5. Lipinski / Veber druglikeness flags (used later in the Streamlit app too)
# For AChE specifically: note that CNS-active drugs need to cross the blood-brain
# barrier, so TPSA <= ~90 and MolWt <= ~450 are tighter, more realistic cutoffs
# than the general Lipinski rule -- both are computed here so you can filter by
# either standard.
df["lipinski_pass"] = (
    (df["MolWt"] <= 500) &
    (df["LogP"] <= 5) &
    (df["HBD"] <= 5) &
    (df["HBA"] <= 10)
)
df["veber_pass"] = (
    (df["RotatableBonds"] <= 10) &
    (df["TPSA"] <= 140)
)
df["cns_mpo_friendly"] = (
    (df["MolWt"] <= 450) &
    (df["TPSA"] <= 90) &
    (df["LogP"] <= 5)
)
df.head()


In [ ]:
# 6. Save and download the cleaned dataset
df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv —", df.shape)

from google.colab import files
files.download("cleaned_data.csv")


**Next step:** open `02_model_training_validation.ipynb` in Colab and upload the `cleaned_data.csv` you just downloaded (or keep both notebooks in the same Colab session / Google Drive folder).